# Phase 2 — Calibration, Generalization, and Real-Time Scheduling
### DAG Execution-Time Quantile Prediction on Heterogeneous Multicore + DVFS

This notebook is a **runnable, narrated walkthrough** of Phase 2. It re-runs the exact
scripts under `scripts/` (each one is also independently runnable from the command line)
and displays their outputs inline. The full written analysis lives in
[`PHASE2_REPORT.md`](../PHASE2_REPORT.md) — this notebook exists for reproducibility and
grading convenience, not to duplicate that discussion in full.

**Primary rule for Phase 2 (unchanged throughout this notebook): the Phase 1 predictor is
frozen.** `best_model.pt` and the exact train-fitted preprocessing state are loaded from
disk; nothing here retrains, fine-tunes, or refits any scaler, and neither the calibration
nor the test_id/test_ood splits are ever used for model selection. The two new baseline
models trained in §5 are independent comparison models — they never touch the frozen
predictor.


In [ ]:
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

RESULTS = PROJECT_ROOT / "results" / "phase2"
PLOTS = RESULTS / "plots"


def run_script(name: str) -> None:
    """Run a Phase 2 script exactly as it would be run from the command line."""
    script_path = PROJECT_ROOT / "scripts" / name
    print(f"--- running {name} ---")
    result = subprocess.run(
        [sys.executable, str(script_path)],
        cwd=str(PROJECT_ROOT), capture_output=True, text=True,
    )
    print(result.stdout[-4000:])
    if result.returncode != 0:
        print(result.stderr[-4000:])
        raise RuntimeError(f"{name} failed with exit code {result.returncode}")


def show(csv_path, n=15):
    df = pd.read_csv(csv_path)
    display(df.head(n))
    return df


def show_plot(name):
    display(Image(filename=str(PLOTS / name)))


## 1. Frozen inference over calibration / test_id / test_ood

Reconstructs the frozen `Phase1QuantilePredictor` (same class definitions extracted from the
Phase 1 notebook), loads `best_model.pt` with strict state-dict loading, and runs inference
over the three reserved splits. Predictions are inverse-transformed to microseconds and
joined with `criticality`/`deadline_us`/`period_us` from `tasks/task_metadata.csv`. See
`src/phase2/inference.py`.


In [ ]:
run_script("run_phase2_inference.py")
raw_test_id = show(RESULTS / "calibration" / "test_id_raw_predictions.csv")


## 2. Split-conformal calibration by criticality (HI / LO)

Fits the one-sided conformal correction (eq. 30-36 of the spec) on the untouched
calibration split, applies the **frozen** correction to test_id and test_ood, and reports
coverage/inflation plus the full α/δ sensitivity sweep. Default operating point:
`base_quantile=Q95`, `δ_HI=0.01`, `δ_LO=0.05` (justification in `src/phase2/config.py` and
§1.1 of the report). See `src/phase2/conformal.py`.


In [ ]:
run_script("run_phase2_calibration.py")
coverage = show(RESULTS / "calibration" / "coverage_by_criticality.csv")
show_plot("coverage_before_after_calibration.png")
show_plot("reliability_diagram.png")
show_plot("inflation_distribution.png")


## 3. OOD generalization + graph-structure sensitivity

Compares test_id vs. test_ood (501-1000 node graphs, never seen at that scale during
training), and breaks accuracy down by core_type, DVFS level, criticality, target role,
graph-size bin, and the generator's structural shape parameters (`fat`, `density_parameter`,
`regular`, each varied in [0.2, 0.8]) using the precomputed
`metadata/graph_summary.csv`. Also derives a critical-path-node proxy and checks whether
coverage degrades specifically on critical-path nodes. See `src/phase2/graph_structure.py`,
`src/phase2/metrics.py`.


In [ ]:
run_script("run_phase2_ood_structure.py")
global_cmp = show(RESULTS / "ood" / "test_id_vs_test_ood_global_metrics.csv")
show_plot("ood_global_comparison.png")
show_plot("accuracy_vs_graph_size.png")
show_plot("critical_path_coverage.png")
show_plot("dvfs_sensitivity.png")


## 4. 4/8/16/32-core scalability stress test

`core_id` was never a model input (only `core_type`), so a literal 32-core hardware
simulation is not something this frozen model can produce. Instead, `active_core_count`
(train range `[1,4]`) is pushed to 4/8/16/32 on real test_id contexts (everything else held
fixed) and the *prediction-uncertainty* shift is reported — deliberately **not** an accuracy
claim, since no ground truth exists for a hypothetical larger system. See
`src/phase2/scalability.py`.


In [ ]:
run_script("run_phase2_scalability.py")
scal = show(RESULTS / "scalability" / "core_scalability_summary.csv")
show_plot("scalability_core_count.png")


## 5. Drift / overload / bursty-state robustness

Three-part study, kept carefully separated between what has real ground truth and what does
not: (a) real in-range tail-load stress (valid accuracy claim), (b) synthetic extreme
overload far outside every training range (descriptive only), and (c) a causal,
strictly-past-only sliding-window conformal calibration compared against the static
correction from §2, testing whether adaptive calibration recovers coverage lost to drift.
See `src/phase2/drift.py`.


In [ ]:
run_script("run_phase2_drift.py")
tail_stress = show(RESULTS / "drift" / "tail_load_stress_accuracy.csv")
overload = show(RESULTS / "drift" / "synthetic_overload_comparison.csv")
sliding = show(RESULTS / "drift" / "static_vs_sliding_window_calibration.csv")
show_plot("static_vs_sliding_window_calibration.png")


## 6. HEFT + deadline-aware scheduling with calibrated budgets

Predicts a cost for **every** node of each test_id DAG (not just the 6 sampled target
nodes/DAG), at a representative DVFS point and the train-fitted mean system state, then
schedules each DAG in isolation on the base 4-core (2 big + 2 little) platform with classic
HEFT and a deadline-partitioned EDF-style deadline-aware list scheduler. Compares three cost
estimators: optimistic Q50, raw uncalibrated Q95, and the calibrated `C_e`. See
`src/phase2/scheduling.py`.


In [ ]:
run_script("run_phase2_scheduling.py")
sched_summary = show(RESULTS / "scheduling" / "scheduling_summary.csv")
show_plot("scheduling_comparison.png")


## 7. Baseline comparison: MLP, GNN-Mean, GNN-Mean-NoZt

Two new, independently-trained comparison models under an explicitly reduced compute budget
(never touching the frozen predictor): an MLP with no graph structure at all, and a GNN-Mean
model sharing the same backbone family as Phase 1 but trained with plain MSE toward the
conditional mean instead of the pinball-loss quantile head. A third z_t-ablated variant
tests offline-vs-online system-state awareness. See `src/phase2/baselines.py`.


In [ ]:
run_script("run_phase2_baselines.py")
model_cmp = show(RESULTS / "baselines" / "model_comparison.csv")
show_plot("model_comparison_mae.png")


## 7b. Deep-dive: operation_type, inference time, GNN-embedding correlation

Closes three specific requirements: (a) grouped accuracy on compute-bound vs. memory-bound
nodes (does the model behave non-linearly across these regimes?), (b) inference time vs.
graph size (training time is N/A -- the predictor is frozen and never retrained per graph
size), and (c) correlation between the frozen GNN's *learned embeddings* (not just a
topological proxy) and prediction accuracy, including on critical-path nodes. See
`src/phase2/embeddings.py`, `scripts/run_phase2_deepdive.py`.


In [ ]:
run_script("run_phase2_deepdive.py")
op_type = show(RESULTS / "ood" / "grouped_test_ood_metrics_by_operation_type.csv")
timing = show(RESULTS / "ood" / "inference_time_vs_graph_size.csv")
embed_corr = show(RESULTS / "ood" / "embedding_error_correlation_test_ood.csv")


## 8. All plots

Regenerates every Phase 2 figure from the CSV artifacts produced above.


In [ ]:
run_script("run_phase2_plots.py")
for name in [
    "violin_predicted_quantiles_by_role.png",
    "violin_predicted_by_hardware_scenario.png",
    "box_actual_vs_predicted.png",
]:
    show_plot(name)


## 9. Conclusion

See [`PHASE2_REPORT.md`](../PHASE2_REPORT.md) for the full written analysis, including the
explicit limitations section (§8) and the artifact index (§9) mapping every claim above to
the CSV/JSON/PNG file that backs it.
